# Module 700 — Data Pipeline Kata
Bronze → Silver → Gold pipeline — K 7.W.1 through K 7.W.7

## K 7.W.1 — Set up your data workspace

In [ ]:
# K 7.W.1 — DuckDB workspace setup
# AI-generated output for the K 7.W.1 setup prompt

import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "duckdb", "pandas", "--quiet"],
    check=True,
)

import duckdb
import pandas as pd
import os
import random
import numpy as np
from datetime import datetime

# In-memory DuckDB connection (reused across all katas in this notebook)
con = duckdb.connect()

# Create hello_world table
con.execute("""
    CREATE TABLE hello_world (
        id        INTEGER,
        message   VARCHAR,
        created_at TIMESTAMP
    )
""")

# Insert 3 sample rows
con.execute("""
    INSERT INTO hello_world VALUES
        (1, 'Hello from DuckDB!',    '2024-01-15 09:00:00'),
        (2, 'Pipeline kata ready.',  '2024-01-15 09:01:00'),
        (3, 'Let the data flow.',    '2024-01-15 09:02:00')
""")

# Query and display
result = con.execute("SELECT * FROM hello_world").fetchdf()
print(result.to_string(index=False))

print("\nEnvironment ready ✓")

## K 7.W.2 — Bronze landing

In [ ]:
# K 7.W.2 — Bronze landing: profile the raw CSV
import duckdb, pandas as pd, os

con = duckdb.connect()

bronze_path = 'bronze/transactions_raw.csv'
bronze_count = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{bronze_path}')").fetchone()[0]
null_amount  = con.execute(
    f"SELECT COUNT(*) FROM read_csv_auto('{bronze_path}', all_varchar=true) "
     "WHERE TRIM(amount) = '' OR amount IS NULL"
).fetchone()[0]

con.execute(f"CREATE TABLE bronze AS SELECT * FROM read_csv_auto('{bronze_path}', all_varchar=true)")

print(f'Bronze rows     : {bronze_count}')
print(f'Null/blank amount: {null_amount}')
print('Bronze table registered in DuckDB ✓')

## K 7.W.3 — Silver cleaning

In [ ]:
# K 7.W.3 — Silver cleaning: remove nulls, dedup, normalise dates

df = con.execute('SELECT * FROM bronze').fetchdf()

def parse_date(s):
    if pd.isna(s): return pd.NaT
    s = str(s).strip()
    for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%b %d %Y'):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            pass
    return pd.NaT

df['order_date_parsed'] = df['order_date'].apply(parse_date)
df['amount_float']      = pd.to_numeric(df['amount'], errors='coerce')

# Remove null amounts (negative amounts are legitimate returns — keep them)
before_null  = len(df)
df           = df[df['amount_float'].notna()]
removed_nulls = before_null - len(df)

# Deduplicate by order_id, keep highest customer_id
before_dedup  = len(df)
df['customer_id_int'] = pd.to_numeric(df['customer_id'], errors='coerce')
df = df.sort_values('customer_id_int', ascending=False).drop_duplicates(subset='order_id', keep='first')
removed_dupes = before_dedup - len(df)
silver_rows   = len(df)

print(f'Removed null amounts      : {removed_nulls}')
print(f'Removed duplicate order_ids: {removed_dupes}')
print(f'Silver rows               : {silver_rows}  (expected 460)')
print(f'Row-count math: {bronze_count} − {removed_nulls} (null) − {removed_dupes} (dupes) = {silver_rows}')
assert silver_rows == 460, f'Expected 460 silver rows, got {silver_rows}'

silver = pd.DataFrame({
    'order_id':         df['order_id'],
    'customer_id':      df['customer_id'],
    'region':           df['region'],
    'order_date':       pd.to_datetime(df['order_date_parsed']).dt.date,
    'product_category': df['product_category'],
    'amount':           df['amount_float'],
    'quantity':         pd.to_numeric(df['quantity'], errors='coerce').astype('Int64'),
    'status':           df['status'],
})

os.makedirs('silver', exist_ok=True)
silver.to_parquet('silver/transactions_clean.parquet', index=False)
print('silver/transactions_clean.parquet written ✓')

## K 7.W.4 — Gold aggregations

In [ ]:
# K 7.W.4 — Gold metrics: daily_sales_by_category + returns_rate

con2 = duckdb.connect()
con2.execute("CREATE TABLE silver AS SELECT * FROM 'silver/transactions_clean.parquet'")

# Gold 1: daily_sales_by_category
# Grain: (order_date, region, product_category)
# Formula: total_revenue = SUM(amount) WHERE status='completed' AND amount > 0
daily_sales = con2.execute("""
    SELECT
        order_date,
        region,
        product_category,
        SUM(amount)              AS total_revenue,
        COUNT(DISTINCT order_id) AS order_count
    FROM silver
    WHERE status = 'completed'
      AND amount > 0
    GROUP BY order_date, region, product_category
""").fetchdf()

grain_dupes = con2.execute("""
    SELECT COUNT(*) - COUNT(DISTINCT CAST(order_date AS VARCHAR)||'|'||region||'|'||product_category)
    FROM (
        SELECT order_date, region, product_category
        FROM silver WHERE status='completed' AND amount > 0
        GROUP BY order_date, region, product_category
    )
""").fetchone()[0]
print(f'daily_sales rows : {len(daily_sales)}, grain dupes: {grain_dupes}')
assert grain_dupes == 0

os.makedirs('gold', exist_ok=True)
daily_sales.to_parquet('gold/daily_sales_by_category.parquet', index=False)
print('gold/daily_sales_by_category.parquet written ✓')

# Gold 2: returns_rate
# Grain: order_date
# Formula: returns_rate_pct = returned / (completed + returned) × 100  (excludes pending)
returns_rate = con2.execute("""
    SELECT
        order_date,
        COUNT(DISTINCT order_id)                                                       AS total_orders,
        COUNT(DISTINCT CASE WHEN status = 'returned' THEN order_id END)               AS returned_orders,
        ROUND(
            100.0 * COUNT(DISTINCT CASE WHEN status = 'returned' THEN order_id END)
                  / NULLIF(COUNT(DISTINCT CASE WHEN status IN ('completed','returned') THEN order_id END), 0),
        2)                                                                             AS returns_rate_pct
    FROM silver
    WHERE status IN ('completed', 'returned')
    GROUP BY order_date
""").fetchdf()

min_rate, max_rate = returns_rate['returns_rate_pct'].min(), returns_rate['returns_rate_pct'].max()
print(f'returns_rate rows: {len(returns_rate)}, rate range: {min_rate} – {max_rate}')
assert min_rate >= 0 and max_rate <= 100

returns_rate.to_parquet('gold/returns_rate.parquet', index=False)
print('gold/returns_rate.parquet written ✓')

## K 7.W.5 — DQ suite: force-tested

In [ ]:
# K 7.W.5 — 8 DQ checks; force-tested with bad-row injection

con3 = duckdb.connect()
con3.execute("CREATE TABLE daily_sales  AS SELECT * FROM 'gold/daily_sales_by_category.parquet'")
con3.execute("CREATE TABLE returns_rate AS SELECT * FROM 'gold/returns_rate.parquet'")

checks = [
    (1, 'No null key columns in daily_sales',
     'SELECT COUNT(*) FROM daily_sales WHERE order_date IS NULL OR region IS NULL OR product_category IS NULL'),
    (2, 'total_revenue > 0',
     'SELECT COUNT(*) FROM daily_sales WHERE total_revenue <= 0'),
    (3, 'order_count > 0',
     'SELECT COUNT(*) FROM daily_sales WHERE order_count <= 0'),
    (4, 'No duplicate grain (order_date, region, product_category)',
     "SELECT COUNT(*) - COUNT(DISTINCT CAST(order_date AS VARCHAR)||'|'||region||'|'||product_category) FROM daily_sales"),
    (5, 'No null order_date in returns_rate',
     'SELECT COUNT(*) FROM returns_rate WHERE order_date IS NULL'),
    (6, 'returns_rate_pct between 0.0 and 100.0',
     'SELECT COUNT(*) FROM returns_rate WHERE returns_rate_pct < 0 OR returns_rate_pct > 100'),
    (7, 'returned_orders <= total_orders',
     'SELECT COUNT(*) FROM returns_rate WHERE returned_orders > total_orders'),
    (8, 'order_date spans at least 30 days',
     "SELECT CASE WHEN DATEDIFF('day', MIN(order_date), MAX(order_date)) < 30 THEN 1 ELSE 0 END FROM returns_rate"),
]

def run_checks(label):
    print(f'=== {label} ===')
    results = []
    for id_, name, sql in checks:
        n = con3.execute(sql).fetchone()[0]
        print(f'  {id_}. {name} → {"PASS" if n == 0 else f"FAIL (violations={n})"}')
        results.append(n)
    return results

clean = run_checks('Clean data')
assert all(n == 0 for n in clean), 'DQ failed on clean data!'
print('Clean pass: 8/8 ✓')

# Inject bad row
con3.execute('INSERT INTO daily_sales VALUES (CURRENT_DATE, \'East\', \'Food\', -999.99, 1)')
print()
injected = run_checks('Bad row injected (total_revenue = -999.99)')
fired = [checks[i][0] for i, n in enumerate(injected) if n > 0]
assert 2 in fired, 'Check 2 (total_revenue > 0) did not fire!'
print(f'Checks that fired: {fired} ✓')

# Remove bad row and re-verify
con3.execute('DELETE FROM daily_sales WHERE total_revenue = -999.99')
print()
post = run_checks('After cleanup')
assert all(n == 0 for n in post), 'DQ failed after cleanup!'
print('Post-cleanup: 8/8 ✓')
print('Break-and-verify complete — checks are trusted gates.')

## K 7.W.6 — Spot-check gold metrics

In [ ]:
# K 7.W.6 — Manual spot-check: verify returns_rate formula against silver

con4 = duckdb.connect()
con4.execute("CREATE TABLE silver      AS SELECT * FROM 'silver/transactions_clean.parquet'")
con4.execute("CREATE TABLE returns_rate AS SELECT * FROM 'gold/returns_rate.parquet'")

spot = con4.execute("""
    SELECT
        r.order_date,
        r.total_orders,
        r.returned_orders,
        r.returns_rate_pct,
        s.total_non_pending AS silver_total,
        s.returned          AS silver_returned
    FROM returns_rate r
    JOIN (
        SELECT
            order_date,
            COUNT(DISTINCT CASE WHEN status IN ('completed','returned') THEN order_id END) AS total_non_pending,
            COUNT(DISTINCT CASE WHEN status = 'returned'               THEN order_id END) AS returned
        FROM silver
        WHERE status IN ('completed', 'returned')
        GROUP BY order_date
    ) s ON r.order_date = s.order_date
    WHERE r.returned_orders > 0
    ORDER BY r.returns_rate_pct DESC
    LIMIT 3
""").fetchdf()

print('Spot-check — formula matches silver source:')
print(spot.to_string(index=False))
print('\nFormula: returned / (completed + returned) × 100 ✓')
print('Pending orders excluded from denominator ✓')